# Clasificador sencillo: TF-IDF + Naive Bayes

Este es el único notebook de modelos del proyecto. Su objetivo es presentar un **baseline fácil de explicar**: toma el título y cuerpo de una noticia, los convierte en números con TF-IDF y entrena un clasificador Multinomial Naive Bayes.

El notebook **no decide la verdad** de una noticia. Solo usa las etiquetas humanas `RESPALDADA` y `REFUTADA` que ya llegaron a Gold después de la revisión en Label Studio.

## Guion breve para exponer

1. Cargamos el CSV Gold validado por personas.
2. Unimos `title + body`: es el único texto que verá el modelo.
3. Separamos 70 % para entrenar y 30 % para probar, manteniendo las clases balanceadas en lo posible.
4. TF-IDF representa las palabras y Naive Bayes aprende qué patrones aparecen en cada clase.
5. Revisamos el reporte y la matriz de confusión.

Claims, PMIDs, similitudes, país y fuente se conservan para auditoría, pero no se usan como variables del modelo para evitar fuga de información.

In [ ]:
# PASO 1. Importar librerías y escoger el archivo Gold.
from collections import Counter
import csv
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Ejecuta el notebook desde la raíz del repositorio.
GOLD_CSV = Path('gold/training/training_20261019.csv')
SEED = 42
# Con el Gold actual, 30 % permite que ambas clases lleguen a la prueba.
TEST_SIZE = 0.30
LABELS = ['REFUTADA', 'RESPALDADA']

print(f'Archivo Gold: {GOLD_CSV.resolve()}')

In [ ]:
# PASO 2. Leer solo las noticias binarias y preparar title + body.
def clean(value):
    return str(value or '').strip()


def load_gold(path):
    if not path.is_file():
        raise FileNotFoundError(f'No existe {path}. Descarga primero el CSV Gold desde OCI.')

    rows, seen_ids = [], set()
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        required = {'record_id', 'title', 'body', 'label'}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'Faltan columnas requeridas: {sorted(missing)}')

        for line_number, source in enumerate(reader, start=2):
            row = {key: clean(value) for key, value in source.items() if key}
            record_id = row.get('record_id', '')
            if not record_id or record_id in seen_ids:
                raise ValueError(f'record_id vacío o repetido en línea {line_number}')
            seen_ids.add(record_id)

            # Dejamos fuera etiquetas no binarias: no son ejemplos para este clasificador.
            if row.get('label') in LABELS:
                if not row.get('title') or not row.get('body'):
                    raise ValueError(f'{record_id} no tiene título o cuerpo completo')
                rows.append(row)
    return rows


rows = load_gold(GOLD_CSV)
class_counts = Counter(row['label'] for row in rows)
print(f'Noticias binarias: {len(rows)}')
print(f'Distribución: {dict(class_counts)}')

if set(class_counts) != set(LABELS) or min(class_counts.values()) < 2:
    raise ValueError('Se necesitan al menos dos noticias de cada clase para hacer un split estratificado.')
if min(class_counts.values()) < 10:
    print('ADVERTENCIA: hay menos de 10 ejemplos en una clase. La ejecución sirve como demostración técnica, no como resultado concluyente.')

texts = [f"{row['title']}\n\n{row['body']}" for row in rows]
labels = [row['label'] for row in rows]

In [ ]:
# PASO 3. Separar datos para entrenar y para probar.
# stratify conserva la proporción de RESPALDADA/REFUTADA en ambos grupos.
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)

if set(y_test) != set(LABELS):
    raise ValueError('La prueba no contiene ambas clases. Se necesitan más ejemplos para evaluar el modelo.')

print(f'Entrenamiento: {len(X_train)} noticias')
print(f'Prueba: {len(X_test)} noticias')
print(f'Clases en prueba: {dict(Counter(y_test))}')

In [ ]:
# PASO 4. Crear y entrenar el baseline.
# TF-IDF convierte palabras y pares de palabras en números; Naive Bayes clasifica esos números.
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1, 2))),
    ('naive_bayes', MultinomialNB()),
])

model.fit(X_train, y_train)
print('Modelo entrenado.')

In [ ]:
# PASO 5. Evaluar una sola vez sobre datos que el modelo no vio al entrenar.
predictions = model.predict(X_test)

print(classification_report(y_test, predictions, labels=LABELS, zero_division=0))
print('Matriz de confusión (filas = real, columnas = predicción):')
print('Orden de clases:', LABELS)
print(confusion_matrix(y_test, predictions, labels=LABELS))

# Importante: estas métricas describen solo este CSV y este split con seed 42.
# No se deben presentar como una conclusión general si una clase tiene pocos ejemplos.

## Cómo explicarlo en una frase

**TF-IDF** transforma el texto en una representación numérica que resalta términos característicos. **Multinomial Naive Bayes** aprende, a partir de las noticias revisadas por personas, qué combinaciones de términos aparecen con mayor frecuencia en cada etiqueta.

Este es un punto de partida reproducible, no un veredicto automático ni un sistema clínico. Para reportar resultados sólidos se necesita ampliar Gold, especialmente la clase `REFUTADA`.